# BFCL V4 — P06 smoke test (official harness)

Purpose: **actually run**, in a user-controlled GPU Colab session, one indexed family across base / missing function / missing parameter with a supported 4B model. This is a **smoke test**, not scientific evidence for a published result. This notebook was authored but **has not been run**. GPU must be enabled in Colab. Notebook pins the official Gorilla source commit; model revision is reported from resolved Hugging Face download information and should be frozen for full experiments. Data and model may require a substantial download.

2026-10-29 troubleshooting revision: BFCL runs in an isolated Python 3.12 environment, with streamed setup/inference/evaluation logs. This revision has not been executed on Colab. If a cell fails, download the corresponding file from /content/P06_BFCL_Logs using Colab's Files panel. Do not continue to generation after setup failure.


In [ ]:
import subprocess, sys, os, json, pathlib, datetime, platform
print('UTC', datetime.datetime.now(datetime.timezone.utc).isoformat())
print('Python', sys.version, 'Platform',platform.platform())
subprocess.run(['nvidia-smi'],check=True)



In [ ]:
from pathlib import Path
root=Path('/content/gorilla')
if not root.exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/ShishirPatil/gorilla.git',str(root)],check=True)
commit='6ea57973c7a6097fd7c5915698c54c17c5b1b6c8'
subprocess.run(['git','-C',str(root),'checkout',commit],check=True)
work=root/'berkeley-function-call-leaderboard'
print('Commit:',subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip())
# BFCL at this commit pins numpy==1.26.4 and vllm==0.8.5.
# Use an isolated interpreter; the Colab notebook kernel may remain Python 3.13.
log_dir=Path('/content/P06_BFCL_Logs')
log_dir.mkdir(exist_ok=True)
def run_logged(cmd, name, cwd=None, env=None):
    log_path=log_dir/(name+'.log')
    print('Running:', ' '.join(map(str,cmd)), flush=True)
    with log_path.open('w') as out:
        proc=subprocess.Popen(list(map(str,cmd)), cwd=cwd, env=env,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            out.write(line)
            out.flush()
        rc=proc.wait()
    if rc:
        print('Saved failure log:', log_path)
        raise subprocess.CalledProcessError(rc, cmd)
    return log_path

run_logged([sys.executable,'-m','pip','install','uv'], 'install_uv')
uv_cmd=[sys.executable,'-m','uv']
run_logged(uv_cmd+['python','install','3.12'], 'install_python312')
venv=Path('/content/bfcl-py312')
if not (venv/'bin/python').exists():
    run_logged(uv_cmd+['venv','--python','3.12','--seed',str(venv)], 'create_venv')
bfcl_python=venv/'bin/python'
bfcl_cli=venv/'bin/bfcl'
run_logged([bfcl_python,'-c','import sys; print(sys.version); assert sys.version_info[:2] == (3,12)'], 'verify_python')
run_logged([bfcl_python,'-m','pip','install','-e',str(work)+'[oss_eval_vllm]'], 'install_bfcl')
run_logged([bfcl_python,'-m','pip','check'], 'pip_check')
run_logged([bfcl_python,'-c','import numpy, torch, vllm; print("numpy", numpy.__version__); print("torch", torch.__version__); print("vllm", vllm.__version__); print("CUDA", torch.cuda.is_available())'], 'verify_imports')
run_logged([bfcl_python,'-m','pip','freeze'], 'pip_freeze')



In [ ]:
config={'multi_turn_base':['multi_turn_base_1'], 'multi_turn_miss_func':['multi_turn_miss_func_1'],'multi_turn_miss_param':['multi_turn_miss_param_1']}
(work/'test_case_ids_to_generate.json').write_text(json.dumps(config,indent=2))
print('Selected smoke-test IDs:',config)
print('Do not extrapolate results from one family.')



In [ ]:
model='Qwen/Qwen3-4B-Instruct-2507-FC'
# Model ID is explicitly listed by the pinned BFCL SUPPORTED_MODELS.md as self-hosted FC.
# Requires compatible GPU and available Colab compute. May fail on constrained runtimes.
os.environ['BFCL_PROJECT_ROOT']=str(work)
cmd=[str(bfcl_cli),'generate','--model',model,'--run-ids','--backend','vllm','--num-gpus','1','--gpu-memory-utilization','0.85']
print('Running:', ' '.join(cmd))
run_env=os.environ.copy()
run_env['PATH']=str(venv/'bin')+os.pathsep+run_env.get('PATH','')
run_logged(cmd, 'generate', cwd=work, env=run_env)



In [ ]:
categories='multi_turn_base,multi_turn_miss_func,multi_turn_miss_param'
cmd=[str(bfcl_cli),'evaluate','--model',model,'--test-category',categories,'--partial-eval']
print('Running:', ' '.join(cmd))
run_env=os.environ.copy()
run_env['PATH']=str(venv/'bin')+os.pathsep+run_env.get('PATH','')
run_logged(cmd, 'evaluate', cwd=work, env=run_env)



In [ ]:
from google.colab import files
import shutil, hashlib
bundle=Path('/content/P06_BFCL_Smoke_Outputs')
bundle.mkdir(exist_ok=True)
if log_dir.exists():
    shutil.copytree(log_dir,bundle/'logs',dirs_exist_ok=True)
for sub in ('result','score'):
    src=work/sub
    if src.exists():
        for path in src.rglob('*'):
            if path.is_file():
                dest=bundle/sub/path.relative_to(src)
                dest.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(path,dest)
manifest={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'bfcl_commit':commit,'model_id':model,'selected_ids':config,'note':'Smoke test, not complete study','files':{}}
for p in bundle.rglob('*'):
    if p.is_file():manifest['files'][str(p.relative_to(bundle))]=hashlib.sha256(p.read_bytes()).hexdigest()
(bundle/'manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive('/content/P06_BFCL_Smoke_Outputs','zip',bundle)
print('Artifacts:',archive,'files:',len(manifest['files']))
files.download(archive)

